# During this notebook, we're implementing a real time training simulation for an ARMA model

In [13]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pandas as pd 

In [14]:
class BackEndARMA:
    def __init__(self, horizon=6, first_train_samples=576, order=(4,3), max_memory=1000):
        self.horizon = horizon
        self.first_train_samples = first_train_samples
        self.order = order
        self.max_memory = max_memory
        self.data = []

    def receive(self, points: list):
        self.data.extend(points)
        self.data = self.data[-self.max_memory:]  # limitar memoria

    def train_and_predict(self):
        if len(self.data) < self.first_train_samples:
            return {"status": "waiting", "message": f"{self.first_train_samples - len(self.data)} samples missing"}
        
        model = SARIMAX(endog=self.data, order=(self.order[0], 1, self.order[1])).fit(disp=False)
        preds = model.forecast(steps=self.horizon)
        return {
            "status": "ok",
            "predictions": preds.tolist()
        }

Now, we simulate it 

In [ ]:
glucose = pd.read_csv('../Data/Preprocessed/HUPA0002P.csv', sep=';')['glucose'] 
backend = BackEndARMA()

for i in range(0, len(glucose), backend.horizon):
    batch = glucose.iloc[i:i+backend.horizon].tolist()
    backend.receive(batch)
    result = backend.train_and_predict()
    print(result)
